# Comparaison 2G vs 1G — Décoding des embeddings protéiques

**Projet Long M2 Bio-Info — Assa Diabira — 2026**

Ce notebook visualise et compare les performances des modèles 2G (ESM-C, Ankh2) et 1G (ESM2, Ankh1G, ProtT5) sur les 6 tâches de prédiction par résidu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path('../..')
RESULTS = ROOT / 'results'
FIGS = RESULTS / 'figures'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

## 1. Chargement des résultats

In [ ]:
df_2g = pd.read_csv(RESULTS / 'dt_results_2g.csv')
df_test = df_2g[df_2g['split'] == 'test'].copy()

# Ajouter colonne génération
gen_map = {'esmc_300m': '2G', 'esmc_600m': '2G', 'ankh2_large': '2G'}
df_test['generation'] = df_test['model'].map(gen_map)

# Labels affichage
model_labels = {
    'esmc_300m': 'ESM-C 300M',
    'esmc_600m': 'ESM-C 600M',
    'ankh2_large': 'Ankh2-Large',
}
df_test['model_label'] = df_test['model'].map(model_labels)

task_order = ['rmsf', 'neq', 'bfact', 'acc', 'sec3', 'sec8']
task_labels = {
    'rmsf': 'RMSF\n(flexibilité)',
    'neq': 'Neq\n(désordre)',
    'bfact': 'Bfactor',
    'acc': 'Accessibilité\nsolvant',
    'sec3': 'SS3',
    'sec8': 'SS8',
}
df_test['task_label'] = df_test['task'].map(task_labels)

print(df_test[['model_label', 'task', 'F1', 'MCC']].pivot_table(
    index='task', columns='model_label', values='F1'
).round(3))

## 2. Comparaison F1 — tous modèles 2G

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['F1', 'MCC']):
    pivot = df_test.pivot_table(index='task', columns='model_label', values=metric)
    pivot = pivot.reindex(task_order)
    pivot.plot(kind='bar', ax=ax, width=0.7)
    ax.set_title(f'{metric} — test set (Decision Tree)', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    ax.set_xticklabels([task_labels[t] for t in task_order], rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.legend(title='Modèle', fontsize=9)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.savefig(FIGS / 'comparison_2g_F1_MCC.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Heatmap F1 — 2G (à compléter avec 1G quand disponible)

In [ ]:
pivot_f1 = df_test.pivot_table(index='task', columns='model_label', values='F1')
pivot_f1 = pivot_f1.reindex(task_order)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_f1, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0.15, vmax=0.9, ax=ax, linewidths=0.5)
ax.set_title('F1 score — test set — modèles 2G', fontsize=13)
ax.set_yticklabels(task_order, rotation=0)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGS / 'heatmap_f1_2g.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Figures PCA

In [ ]:
from IPython.display import Image, display
import os

pca_figs = sorted(FIGS.glob('*_pca_*.png'))
print(f'{len(pca_figs)} figures PCA trouvées')

for f in pca_figs[:6]:
    print(f.name)
    display(Image(str(f), width=500))

## 5. Comparaison 2G vs 1G (à compléter après génération baselines)

In [ ]:
# Charger les résultats 1G quand disponibles
results_1g_path = RESULTS / 'dt_results_1g.csv'

if results_1g_path.exists():
    df_1g = pd.read_csv(results_1g_path)
    df_all = pd.concat([df_2g, df_1g])
    df_all_test = df_all[df_all['split'] == 'test']
    
    pivot_all = df_all_test.pivot_table(index='task', columns='model', values='F1')
    print(pivot_all.round(3))
else:
    print('Résultats 1G pas encore disponibles — lancer scripts/run_embeddings_1g.sh')
    print('puis build_1g_datasets.sh et run_analyses_1g.sh')